In [56]:
import pandas as pd
import numpy as np

In [57]:
df = pd.read_csv("student_performance.csv")

In [58]:
df =df.drop(columns = ['math_score','science_score','english_score','final_grade','student_id','age'])

In [59]:
df.head()

,gender,school_type,parent_education,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method,overall_score
0,male,public,post graduate,3.1,84.3,yes,<15 min,yes,notes,53.1
1,female,public,graduate,3.7,87.8,yes,>60 min,no,textbook,61.3
2,female,private,post graduate,7.9,65.5,no,<15 min,no,notes,89.6
3,other,public,high school,1.1,58.1,no,15-30 min,no,notes,41.6
4,female,public,high school,1.3,61.0,yes,30-60 min,yes,group study,25.4


In [60]:
df = df.drop_duplicates()

In [61]:
df.isnull().sum()

gender                   0
school_type              0
parent_education         0
study_hours              0
attendance_percentage    0
internet_access          0
travel_time              0
extra_activities         0
study_method             0
overall_score            0
dtype: int64

In [62]:
df.duplicated().sum()

np.int64(0)

# train test split

In [63]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['overall_score']),df['overall_score'],
                                                test_size=0.2)

In [64]:
X_train

,gender,school_type,parent_education,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method
2897,female,public,no formal,3.1,83.9,yes,<15 min,yes,coaching
7112,male,private,diploma,3.4,76.4,yes,<15 min,yes,textbook
10229,male,private,post graduate,6.4,76.7,yes,15-30 min,yes,textbook
8978,other,public,no formal,1.6,70.1,no,<15 min,no,online videos
4834,male,public,post graduate,7.1,57.2,no,>60 min,no,group study
...,...,...,...,...,...,...,...,...,...
12627,other,public,no formal,0.9,86.7,yes,>60 min,no,online videos
9491,female,private,post graduate,6.3,89.2,yes,15-30 min,yes,online videos
1834,other,public,phd,4.4,94.7,yes,<15 min,no,notes
13349,male,public,phd,1.9,57.4,yes,15-30 min,no,group study


# column transformer

In [65]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler


In [66]:
ordinal_cols = ['parent_education','travel_time']
nominal_cols = ['gender','school_type','internet_access','extra_activities','study_method']
numeric_cols = ['study_hours','attendance_percentage']
#order
categories = [
    ['no formal','high school','diploma','graduate','post graduate','phd'],
    ['<15 min','15-30 min','30-60 min','>60 min']
]
#transformer
preprocessor = ColumnTransformer(transformers =[
    ('ord',OrdinalEncoder(categories=categories),ordinal_cols),
    ('ohe',OneHotEncoder(drop = 'first',sparse_output = False),nominal_cols),
    ('num',StandardScaler(),numeric_cols)
])

In [67]:
preprocessor.fit_transform(X_train).shape

(12000, 14)

In [68]:
preprocessor.fit_transform(X_test).shape

(3000, 14)

# Random Forest Regressor

In [69]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Pipeline

In [70]:
pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model",
    RandomForestRegressor(n_estimators=100))
] )

In [71]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('ord',
                                                  OrdinalEncoder(categories=[['no '
                                                                              'formal',
                                                                              'high '
                                                                              'school',
                                                                              'diploma',
                                                                              'graduate',
                                                                              'post '
                                                                              'graduate',
                                                                              'phd'],
                                                                             ['<15 '
                                                                              'min',
                                                                              '15-30 '
                                                                              'min',
                                                                              '30-60 '
                                                                              'min',
                                                                              '>60 '
                                                                              'min']]),
                                                  ['parent_education',
                                                   'travel_time']),
                                                 ('ohe',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['gender', 'school_type',
                                                   'internet_access',
                                                   'extra_activities',
                                                   'study_method']),
                                                 ('num', StandardScaler(),
                                                  ['study_hours',
                                                   'attendance_percentage'])])),
                ('model', RandomForestRegressor())])

# r2_score

In [72]:
from sklearn.metrics import r2_score
y_pred = pipeline.predict(X_test)
print(r2_score(y_test, y_pred))

0.8975760489634644


# mean_absolute_error

In [73]:
from sklearn.metrics import mean_absolute_error
print("MAE:",mean_absolute_error(y_test,y_pred))

MAE: 5.097791583333334


# mean_squared_error

In [74]:
from sklearn.metrics import mean_squared_error
print("MSE:",mean_squared_error(y_test,y_pred))

MSE: 35.881086417812504


In [75]:
import numpy as np
from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
print("RMSE:",rmse)

RMSE: 5.99008233814966


In [76]:
import joblib

In [77]:
joblib.dump(pipeline,'model.pkl')
model = joblib.load('model.pkl')

In [78]:
import pandas as pd
sample = pd.DataFrame([[
    'other','private','post graduate',5.0,94.4,'yes','15-30 min','yes','coaching'
]],columns = [
'gender','school_type','parent_education','study_hours',
    'attendance_percentage','internet_access','travel_time','extra_activities','study_method'
    
])

In [79]:
model.predict(sample)

array([74.469])

In [80]:
import sklearn
print(sklearn.__version__)

1.6.1
